# Creating an A2A Health Research Agent using OpenAI Agents SDK

In this exercise, you will build a second agent: a Health Research Agent. Unlike the Policy Agent which used PDF documents, this agent will use OpenAI's Agents SDK and a tool to search the web for health information. You will also see how to easily wrap an Agents SDK agent into an A2A server.

## Define the Research Agent

You will write the code for `a2a_research_agent.py`:
- **OpenAI Agents SDK**: You will use `Agent` from `agents`, which simplifies agent creation.
- **Tools**: You will equip the agent with `WebSearchTool` to allow it to fetch external information.
- **A2A Integration**: Instead of manually defining the Executor and RequestHandler as in Lesson 4, you will use `google.adk.a2a.utils.agent_to_a2a.to_a2a` to automatically wrap the ADK agent into an A2A-compliant application.

In [5]:
from agents import Agent, WebSearchTool, Runner, RunConfig, set_default_openai_client
from openai import AsyncOpenAI
import asyncio

client = AsyncOpenAI(
    base_url="http://172.16.0.237:8004/v1",
    api_key= "gates-digidigidigidong"    
)

set_default_openai_client(client)

INSTRUCTIONS = """You are a healthcare research agent tasked with providing information about health conditions.
Use the WebSearchTool to find information on the web about options, symptoms, treatments, and procedures.
Cite your sources in your responses.
Output all of the information you find. 
"""

research_agent = Agent(
    name="HealthResearchAgent",
    model="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools = [WebSearchTool()],
)

In [7]:
result = await Runner.run(
    research_agent, 
    "How do I get mental health therapy?",
    run_config=RunConfig(
        tracing_disabled=True
    )
)
print(result.final_output)

Seeking mental health therapy is a commendable step toward improving your well-being. Here's a comprehensive guide to help you navigate the process:

**1. Identify Your Needs and Goals**

Begin by reflecting on the specific challenges you're facing and what you hope to achieve through therapy. Understanding your objectives will assist in finding a therapist who aligns with your needs.

**2. Explore Available Resources**

- **Online Directories:** Utilize reputable platforms to search for therapists in your area. The Substance Abuse and Mental Health Services Administration (SAMHSA) offers a confidential and anonymous resource for individuals seeking treatment for mental and substance use disorders in the United States and its territories. ([findtreatment.gov](https://findtreatment.gov/?utm_source=openai))

- **Professional Organizations:** Organizations such as the American Psychological Association (APA) and the National Association of Social Workers (NASW) provide directories of lice

## Refactor into an Agent Class

To make this code reusable and easier to integrate into an A2A server later, you will wrap the logic into a `HealthResearchAgent` class in a file named `agents.py`. This class initializes the client and exposes an `answer_query` method.

In [2]:
%%writefile health_research_agent.py
from agents import Agent, WebSearchTool, Runner, RunConfig, set_default_openai_client
from openai import AsyncOpenAI
import asyncio


class HealthResearchAgent:
    def __init__(self) -> None:
        self.client = AsyncOpenAI(
            base_url="http://172.16.0.237:8004/v1",
            api_key= "gates-digidigidigidong"    
        )
           
        set_default_openai_client(self.client)
        
        INSTRUCTIONS = """You are a healthcare research agent tasked with providing information about health conditions.
        Use the WebSearchTool to find information on the web about options, symptoms, treatments, and procedures.
        Cite your sources in your responses.
        Output all of the information you find. 
        """
        self.research_agent = Agent(
            name="HealthResearchAgent",
            model="gpt-4o-mini",
            instructions=INSTRUCTIONS,
            tools = [WebSearchTool()],
        )

    async def answer_query(self, prompt: str) -> str:
        result = await Runner.run(
            self.research_agent, 
            prompt,
            run_config=RunConfig(
                tracing_disabled=True
            )
        )
        return result.final_output

Overwriting health_research_agent.py


## Test the Agent Class

Finally, import the `HealthResearchAgent` class you just created and test it with the same query to ensure it works as expected.

In [1]:
from health_research_agent import HealthResearchAgent
from IPython.display import Markdown, display

print("Running Health Research Agent")
agent = HealthResearchAgent()
prompt = "How do I get mental health therapy?"

response = await agent.answer_query(prompt)
display(Markdown(response))

Running Health Research Agent


Seeking mental health therapy is a commendable step toward improving your well-being. Here's a guide to help you navigate the process:

**1. Assess Your Needs and Goals**

Begin by identifying the specific challenges or symptoms you're experiencing. Determine whether you prefer therapy, medication, or a combination of both. Understanding your objectives will guide you in selecting the most suitable treatment approach. ([healthline.com](https://www.healthline.com/health/mental-health-resources?utm_source=openai))

**2. Explore Available Resources**

- **Insurance Coverage:** Contact your health insurance provider to inquire about mental health services covered under your plan. They can provide a list of in-network therapists and facilities. ([healthline.com](https://www.healthline.com/health/mental-health-resources?utm_source=openai))

- **Community Health Centers:** Federally Qualified Health Centers (FQHCs) offer comprehensive medical services, including mental health counseling, often on a sliding fee scale based on income. Use the Health Resources and Services Administration database to locate centers near you. ([healthline.com](https://www.healthline.com/health/mental-health/mental-health-services?utm_source=openai))

- **Online Directories:** Utilize reputable online directories to find mental health professionals in your area. Websites like the Substance Abuse and Mental Health Services Administration's (SAMHSA) treatment locator and the American Psychological Association's Psychologist Locator can be valuable resources. ([psychcentral.com](https://psychcentral.com/health/find-help?utm_source=openai))

**3. Consider Telehealth Options**

Online therapy platforms, such as Talkspace and BetterHelp, connect you with licensed therapists through secure messaging and video sessions. These services offer flexibility and can be more accessible, especially if in-person visits are challenging. ([healthline.com](https://www.healthline.com/health/mental-health/mental-health-services?utm_source=openai))

**4. Seek Recommendations**

Reach out to trusted individuals—friends, family, or healthcare providers—for therapist recommendations. Personal referrals can provide insights into a therapist's approach and effectiveness. ([samhsa.gov](https://www.samhsa.gov/find-support/health-care-or-support?utm_source=openai))

**5. Prepare for Initial Contact**

When reaching out to potential therapists, be ready to discuss:

- Your specific concerns and goals

- Insurance details or financial considerations

- Preferred therapy modalities (e.g., cognitive-behavioral therapy, psychodynamic therapy)

- Availability and scheduling preferences

This preparation will help you find a therapist who aligns with your needs. ([samhsa.gov](https://www.samhsa.gov/find-support/health-care-or-support/how-to-set-up-an-appointment?utm_source=openai))

**6. Understand Your Rights and Costs**

Familiarize yourself with the Mental Health Parity and Addiction Equity Act, which mandates that insurance coverage for mental health services be comparable to that for medical services. This ensures that your insurance plan cannot impose higher co-pays or stricter limits on mental health care. ([kiplinger.com](https://www.kiplinger.com/personal-finance/health-insurance/managing-the-high-cost-of-mental-health-care?utm_source=openai))

**7. Be Persistent and Patient**

Finding the right therapist may take time. It's essential to feel comfortable and understood by your therapist. If the first match doesn't feel right, consider exploring other options. Remember, seeking help is a courageous and vital step toward better mental health.

By utilizing these resources and approaches, you can find a mental health therapist who suits your needs and supports your journey toward well-being. 

## Wrapping the Health Research Agent into an A2A Server

In [3]:
%%writefile health_research_a2a.py
import os
import uvicorn

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)
from a2a.utils import new_agent_text_message

from health_research_agent import HealthResearchAgent


class HealthResearchAgentExecutor(AgentExecutor):
    def __init__(self) -> None:
        self.agent = HealthResearchAgent()

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        prompt = context.get_user_input()
        response = await self.agent.answer_query(prompt)
        message = new_agent_text_message(response)
        await event_queue.enqueue_event(message)

    async def cancel(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        pass

def main() -> None:
    print(f"Running A2A Health Research Agent")

    PORT = 9998
    HOST = "localhost"

    skill = AgentSkill(
        id="health_research",
        name="Health Research",
        description="Searches the web for healthcare information about symptoms, health conditions, treatments, and procedures using up-to-date web resources.",
        tags=["symptoms", "health conditions", "treatments", "medical procedures"],
        examples=["How do I get mental health therapy", "What are the symptoms of covid", "home remedies for cough"],
    )

    agent_card = AgentCard(
        name="HealthResearchAgent",
        description="Provides healthcare information about symptoms, health conditions, treatments, and procedures using up-to-date web resources.",
        url=f"http://{HOST}:{PORT}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=HealthResearchAgentExecutor(),
        task_store=InMemoryTaskStore(),
    )

    server = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler,
    )

    uvicorn.run(server.build(), host=HOST, port=PORT)

    
if __name__ == '__main__':
    main()
        

Overwriting health_research_a2a.py


## Run the Health Research A2A Server

Now to activate your configured A2A agent, you would need to run your agent server. You can run the agent server using `uv`:

- Open a terminal
- Navigate to the `3-HealthResearchAgent` directory:
    - `cd 3-HealthResearchAgent`
    - `uv init`
- Activate the virtual environment:
    - `uv venv`
    - `source .venv/bin/activate`
- Install the additional dependencies:
    - `uv add "a2a-sdk<1.0" openai requests uvicorn starlette sse-starlette openai-agents`
- Type `uv run a2a_policy_agent.py` to run the server and activate your A2A agent. 
- When running the agent for the first time, you will see that the virtual environment will first be created automatically and then the agent will run.